<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c42-header.png" alt="Nextia Learning · Vision AI: See, Count and Inspect" width="100%">

# Solution: Domain shift

**[Domain shift](https://learning.nextia-ai.com/courses/vision/m08/domain-shift/)** · Vision AI: See, Count and Inspect · Module 8, lesson 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** find a domain shift (a new studio light, a new camera) in the photos themselves and in a model's scores, measure it with the population stability index, and try the cheapest fix.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells. No account, no key and no model download (the scores were recorded). |
| **You should know** | Pass, check and fail lines, from [Set the line with the warehouse](https://learning.nextia-ai.com/courses/vision/m06/set-the-line-with-the-warehouse/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, Pillow 13.0.0, scikit-learn 1.9.1 and matplotlib 3.11.2 (PyTorch 2.15.0 and timm 1.0.31 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/vision/m08/domain-shift/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C42/M08-L01-domain-shift/domain-shift-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, pillow 13.0.0, scikit-learn 1.9.1, scikit-image 0.26.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 pillow==13.0.0 scikit-learn==1.9.1 scikit-image==0.26.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import numpy; print("numpy", numpy.__version__)
import pandas; print("pandas", pandas.__version__)
import PIL; print("pillow", PIL.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import skimage; print("scikit-image", skimage.__version__)

## Setup: get the Larkfield images

Run the next cell. It downloads the image sets that this lesson needs (about 34 MB, as zip files) into a folder `larkfield-images`, with the course's helper `compare.py`, checks each file's checksum and unpacks it. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-images` (in Colab: the **Files** panel on the left), then run the cell again.

Where the photos come from: the catalogue photos are cut from Open Images photos (CC BY 2.0: the author of each photo is in `data/catalogue/manifest.csv`), the returned candles are photos of the VisA inspection set (CC BY 4.0), and the pallets are drawn by our code. Read `data/dataset-card.md` for the details.

In [ ]:
# Setup: download the image sets into larkfield-images/, check them and unpack them.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C42/"
FILES = {
    "archives/catalogue.zip": "449365badb665189b8bdc93d2ebb6f8d861d8ef1627d1c1a548e996960b89bda",
    "archives/returns.zip": "dcf02f521f68228d34717b303628b2db8b19d18433126da2ce64047f1449210f",
    "archives/extras.zip": "c2d0ecd8e6fda3846550f27aa054877a0f9daa4f6934d315907ecf107427a104",
    "compare.py": "95b41e2ed829d1e1be658947c266f53d571d9b441c4ed228d2cdec2750f4ed50",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-images").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-images and run again.")
    if name.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(PROJECT / "data")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
n = sum(1 for p in Path("data").rglob("*") if p.is_file())
print(f"Ready: {n} files in larkfield-images/data.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, Pillow's `Image`, matplotlib, and the course helper as `cp` (the comparison table). It defines `show(paths)`, which draws photos in a row, and the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: images, the comparison table, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import compare as cp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
CLASSES = ["bowl", "candle", "chair", "cushion", "lamp", "mug", "plant_pot", "vase"]

def show(paths, titles=None, size=2.2):
    """Draw photos in a row (at most 8)."""
    paths = list(paths)[:8]
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size))
    for i, (ax, p) in enumerate(zip(np.atleast_1d(axes), paths)):
        ax.imshow(Image.open(p)); ax.axis("off")
        if titles:
            ax.set_title(str(titles[i]), fontsize=8)
    plt.show()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The new studio

The catalogue team moved to a new studio with warmer light. `data/catalogue/new_studio/` has the 160 test photos again, as the new studio would take them (a change that our code made).

In [ ]:
catalogue = pd.read_csv("data/catalogue/manifest.csv")
studio = pd.read_csv("data/catalogue/new_studio_manifest.csv")
test = catalogue[catalogue["split"] == "test"].reset_index(drop=True)
show(["data/" + test["file"][3], "data/" + studio["file"][3]], ["old studio", "new studio"], size=3)

def channel_means(files):
    return np.stack([np.asarray(Image.open("data/" + f).convert("RGB")).reshape(-1, 3).mean(0) for f in files])
print("mean red, green, blue: old", channel_means(test["file"]).mean(0).round(1), "new", channel_means(studio["file"]).mean(0).round(1))

The course's table shows what the new light does to each method: the colour histogram loses about 4 points; HOG (no colour) loses nothing; CLIP's probe stays near 0.9.

In [ ]:
table = cp.Table.load("data/comparison_table.json")
rows = [r for r in table.rows if r["task"] == "catalogue" and r["setting"] in ("catalogue test", "new studio")]
pd.DataFrame(rows).pivot_table(index="method", columns="setting", values="value").round(3).head(12)

## 2. The new returns camera

Kwame's team replaced the returns camera in the middle of a month. `data/returns/month/` has 30 days of photos (no labels). The course team recorded the scores of the project's inspector (a small PatchCore) for every photo.

> **Predict.** What happens to the anomaly scores after the camera change: up, down or the same?

In [ ]:
month = pd.read_csv("data/returns/month_manifest.csv")
returns = pd.read_csv("data/returns/manifest.csv")
scores = json.load(open("data/predictions/returns_small_patchcore.json"))
month["score"] = month["item_id"].map(scores)
month["day_number"] = month["day"].str[-2:].astype(int)
reference = [scores[i] for i in returns[(returns["split"] == "val") & (returns["status"] == "good")]["item_id"]]
plt.figure(figsize=(9, 3)); plt.scatter(month["day_number"], month["score"], s=10)
plt.axhline(np.mean(reference), color="gray", ls="--", label="mean of good validation photos")
plt.xlabel("day of the month"); plt.ylabel("anomaly score"); plt.legend(); plt.show()
show(list("data/" + month[month["day_number"] == 14]["file"][:2]) + list("data/" + month[month["day_number"] == 17]["file"][:2]),
     ["day 14", "day 14", "day 17", "day 17"], size=2.8)

> **Check.** The mean score is 7.65 on days 1–15 and 14.21 on days 16–30. Every photo after the change looks "unusual" to the inspector: it fails them all. The model did not get worse at seeing damage; the photos changed.

## 3. Measure the move: PSI

The population stability index compares two distributions of a number. Below 0.1 is small; above 0.25 is a large move.

In [ ]:
def psi(expected, actual, bins=10):
    """Population stability index: 0 = the same distribution; above 0.25 = a large move."""
    e, a = np.asarray(expected, float), np.asarray(actual, float)
    edges = np.unique(np.quantile(e, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    pe = np.clip(np.histogram(e, edges)[0] / len(e), 1e-4, None)
    pa = np.clip(np.histogram(a, edges)[0] / len(a), 1e-4, None)
    return float(((pa - pe) * np.log(pa / pe)).sum())

early = month[month["day_number"] <= 15]["score"]
print("PSI, days 1-15 against validation:", round(psi(reference, early), 2))

> **Check.** About 0.2 for days 1–15: a moderate move, under the 0.25 line, from a small sample (75 photos, with damaged ones in them). The service's weekly check alarms on one of these weeks: a false alarm.

## 4. Your turn: the PSI after the change

> **Your turn.** Put the PSI of the scores of days 16–30 against the good validation scores (`reference`) in `psi_late` (a number).

In [ ]:
late = month[month["day_number"] >= 16]["score"]
psi_late = psi(reference, late)
print(round(psi_late, 2))

In [ ]:
expect('psi_late (rounded)', None if psi_late is ... else round(psi_late, 2), 8.28, tolerance=0.01)

## 5. The cheapest fix

Three fixes, from cheap to expensive: a **camera rule** (fix the white balance and the exposure, or correct each photo's colours by code); **new good photos** of the new camera in the inspector's memory; **new labels** and new lines. The course measured the first two on days 16–30: the colour correction alone did almost nothing (AUROC 0.54); 30 checked good photos in the memory helped (AUROC 0.68) but not enough. Write two sentences for Kwame: what do you do on day 16?

## Next

The [lesson page](https://learning.nextia-ai.com/courses/vision/m08/domain-shift/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Fast enough on a CPU](https://learning.nextia-ai.com/courses/vision/m08/fast-enough-on-a-cpu/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/vision/m08/domain-shift/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The photos are open photos (each with its author and licence in the manifests) or images made by our code; no photo shows a real person.